In [ ]:
# 相场数据流水线 — 改RUN_START/RUN_END一口气跑多组 (Kaggle)
import os
import numpy as np
import cupy as cp
from tqdm import tqdm
from scipy import ndimage

# =======================================================================
# 时间步敏感性：512² @ 2 nm，dt=2.5e-4；与基准推进到相同 T*=252
# =======================================================================
nx, ny = 512, 512
dx_real = 2e-9
dt = 2.5e-4
total_steps = 1008000
save_interval = 4000

c_gamma  = 0.10
c_gprime = 0.22
c_matrix = 0.13

gamma_int = 0.091
gamma_gb  = 1.02
delta     = 0.0072
C11 = 174.9e9
C12 = 117.6e9
C44 = 71.6e9

f0 = 1.27e9
dx = dx_real
lam = 5.0
dc = c_gprime - c_gamma

koA = (lam * dc / 4) ** 2
gid = gamma_int / (f0 * dx)
gbd = gamma_gb  / (f0 * dx)
ktA = (gid * 6 / dc**3) ** 2
A_well  = np.sqrt(ktA / koA)
kappa_c = koA * A_well

D_bulk = 10.0
D_gb   = 30.0
S_stab = D_bulk * A_well
elastic_scale = 150.0
GB_penalty = 50.0

alpha_pinning = 25.0
beta_zener = 1500.0

W_gb = 1000.0
kappa_eta = 160.0
L_mobility = 0.4
num_grains = 6

assert abs(total_steps * dt - 252.0) < 1e-12
assert abs(save_interval * dt - 1.0) < 1e-12

# =======================================================================
# ★ 改这两个数字一口气跑多组
# =======================================================================
RUN_START = 0
RUN_END   = 0

for run_id in range(RUN_START, RUN_END + 1):
    output_dir = "/kaggle/working/Temporal_dt2p5e-4_Run_{:02d}".format(run_id)
    os.makedirs(output_dir, exist_ok=True)
    print("=" * 60)
    print("Run {:02d} -> {}".format(run_id, output_dir))

    # ---- 初始组织: Voronoi + 独立高斯随机场 ----
    grain_seeds = [(100,150),(300,100),(450,220),(380,420),(180,400),(80,280)]
    yg, xg = cp.indices((ny, nx))

    _kx = cp.fft.fftfreq(nx) * 2 * np.pi
    _ky = cp.fft.fftfreq(ny) * 2 * np.pi
    _K2p = _kx[:, None]**2 + _ky[None, :]**2

    cp.random.seed(777 + run_id)
    _nx = (cp.random.randn(ny, nx) + 1j * cp.random.randn(ny, nx))
    _nx = _nx * cp.exp(-_K2p * 60**2 / 2)
    _fx = cp.real(cp.fft.ifft2(_nx))
    _fx *= 25.0 / cp.std(_fx)

    cp.random.seed(888 + run_id)
    _ny = (cp.random.randn(ny, nx) + 1j * cp.random.randn(ny, nx))
    _ny = _ny * cp.exp(-_K2p * 60**2 / 2)
    _fy = cp.real(cp.fft.ifft2(_ny))
    _fy *= 25.0 / cp.std(_fy)

    xgw = xg + _fx
    ygw = yg + _fy

    ds = []
    for sx, sy in grain_seeds:
        dx = xgw - sx
        dx = dx - cp.round(dx / nx) * nx
        dy = ygw - sy
        dy = dy - cp.round(dy / ny) * ny
        ds.append(dx**2 + dy**2)
    gid = cp.argmin(cp.array(ds), axis=0)

    etas = cp.zeros((num_grains, ny, nx))
    for i in range(num_grains):
        etas[i] = cp.where(gid == i, 1.0, 0.0)
    for i in range(num_grains):
        etas[i] = cp.real(cp.fft.ifft2(
            cp.fft.fft2(etas[i]) * cp.exp(-_K2p * 1.0**2 / 2)))

    # ---- 撒种 ----
    cp.random.seed(42 + run_id * 7)
    c = cp.full((ny, nx), c_matrix)

    gbx = (gid != cp.roll(gid, 1, axis=1))
    gby = (gid != cp.roll(gid, 1, axis=0))
    gbm = gbx | gby
    gbi = cp.where(gbm)
    ng  = len(gbi[0])
    cp.random.seed(99 + run_id * 10)
    ch = cp.random.choice(cp.arange(ng), size=60, replace=False)
    for k in ch:
        iy = int(gbi[0][k])
        ix = int(gbi[1][k])
        ow = int(gid[iy, ix])
        off = [(-1,0),(1,0),(0,-1),(0,1),(-1,-1),(1,-1),(-1,1),(1,1)]
        bd = 0
        bx = 0
        by = 0
        for dx, dy in off:
            ct = sum(1 for s in range(1, 8)
                      if 0 <= iy + s*dy < ny
                      and 0 <= ix + s*dx < nx
                      and int(gid[iy + s*dy, ix + s*dx]) == ow)
            if ct > bd:
                bd = ct
                bx = dx
                by = dy
        cx = ix + bx * 6
        cy = iy + by * 6
        YY, XX = cp.meshgrid(
            cp.clip(cp.arange(ny) - cy, -12, 12),
            cp.clip(cp.arange(nx) - cx, -12, 12),
            indexing='ij')
        c = cp.where((gid == ow) & (XX**2 + YY**2 < 16), c_gprime, c)

    cp.random.seed(123 + run_id * 13)
    for g in range(num_grains):
        mg = (gid == g)
        gyi, gxi = cp.where(mg)
        for _ in range(60):
            idx = int(cp.random.randint(0, len(gxi), 1)[0])
            sx = int(gxi[idx])
            sy = int(gyi[idx])
            c = cp.where(
                (gid == g) & ((xg - sx)**2 + (yg - sy)**2 < 4),
                c_gprime, c)

    # ---- 弹性核 B(k) ----
    kx = cp.fft.fftfreq(nx) * 2 * np.pi
    ky = cp.fft.fftfreq(ny) * 2 * np.pi
    KX, KY = cp.meshgrid(kx, ky)
    K2 = KX**2 + KY**2
    K2s = cp.where(K2 == 0, 1.0, K2)
    K4r = (KX**4 + KY**4) / K2s**2

    AZ = 2 * C44 / (C11 - C12)
    an = (AZ - 1) / (AZ + 1)
    B0 = elastic_scale * (C11 / f0) * delta**2

    ga = [0.0, np.pi/6, np.pi/3, np.pi/2, 2*np.pi/3, 5*np.pi/6]
    BK = cp.zeros((num_grains, ny, nx))
    for idx, th in enumerate(ga):
        KXr = KX * cp.cos(th) + KY * cp.sin(th)
        KYr = -KX * cp.sin(th) + KY * cp.cos(th)
        K4rr = (KXr**4 + KYr**4) / K2s**2
        Bk = -B0 * (1.0 + an * K4rr)
        Bk[0, 0] = 0.0
        BK[idx] = Bk

    # ---- 预计算 ----
    ak4 = K2**2
    lhs_e = 1.0 + dt * L_mobility * kappa_eta * K2
    lhs_c = 1.0 + dt * S_stab * ak4

    def get_phi(etas_in):
        return cp.clip((1.0 - cp.sum(etas_in**2, axis=0)) * 2.0, 0.0, 1.0)

    phi = get_phi(etas)
    Deff = D_bulk * (1 - phi) + D_gb * phi

    def laplacian(f):
        return cp.real(cp.fft.ifft2(-K2 * cp.fft.fft2(f)))

    # 保存未推进的共同初态，供离散设置之间核对；不参与时间推进。
    initial7 = np.concatenate([
        cp.asnumpy(c).astype(np.float32)[None, :, :],
        cp.asnumpy(etas).astype(np.float32)], axis=0)
    np.save(os.path.join(output_dir, 'initial_state_T000_float32.npy'), initial7)
    del initial7

    t_hist = []
    r_hist = []
    v_hist = []
    l_hist = []

    # ---- 主循环 ----
    for step in tqdm(range(1, total_steps + 1), desc="Run{:02d}".format(run_id)):

        # Cahn-Hilliard
        fc = A_well * (c - c_gamma) * (c - c_gprime) * (2*c - c_gamma - c_gprime)
        cf = cp.fft.fft2(c)
        dE = cp.zeros((ny, nx))
        for i in range(num_grains):
            dE += cp.real(cp.fft.ifft2(BK[i] * cf)) * etas[i]
        mu = fc - kappa_c * laplacian(c) + dE + GB_penalty * phi * (c - c_gamma)
        Dx = 0.5 * (Deff + cp.roll(Deff, -1, 1))
        fx = Dx * (cp.roll(mu, -1, 1) - mu)
        Dy = 0.5 * (Deff + cp.roll(Deff, -1, 0))
        fy = Dy * (cp.roll(mu, -1, 0) - mu)
        df = (fx - cp.roll(fx, 1, 1)) + (fy - cp.roll(fy, 1, 0))
        cf = cp.fft.fft2(c)
        rc = cf + dt * cp.fft.fft2(df) + dt * S_stab * ak4 * cf
        c = cp.real(cp.fft.ifft2(rc / lhs_c))
        c += (c_matrix - cp.mean(c))
        c = cp.clip(c, 0.08, 0.24)

        # Allen-Cahn + Zener
        Wgv = W_gb * (1.0 + alpha_pinning * cp.clip(c - c_gamma, 0, 1))
        zf = beta_zener * cp.clip(c - c_gamma, 0, 1) ** 2
        ss = cp.sum(etas**2, axis=0)
        for i in range(num_grains):
            sj = ss - etas[i]**2
            sw = etas[i] * (etas[i]**2 - 1) + 2 * etas[i] * sj
            sz = zf * etas[i] * (1 - etas[i]) * (etas[i] - 0.5)
            ef = cp.fft.fft2(etas[i])
            sf = cp.fft.fft2(-L_mobility * (Wgv * sw + sz))
            etas[i] = cp.real(cp.fft.ifft2((ef + dt * sf) / lhs_e))
            etas[i] = cp.clip(etas[i], 0.0, 1.0)
        phi = get_phi(etas)
        Deff = D_bulk * (1 - phi) + D_gb * phi

        # 保存 + 诊断 (7ch: c + eta1 + ... + eta6)
        if step % save_interval == 0:
            if not bool(cp.all(cp.isfinite(c))) or not bool(cp.all(cp.isfinite(etas))):
                raise FloatingPointError(f'Non-finite field detected at step {step}')
            cn = cp.asnumpy(c).astype(np.float16)
            en = cp.asnumpy(etas).astype(np.float16)
            frame7 = np.concatenate([cn[None, :, :], en], axis=0)
            assert frame7.shape == (7, ny, nx)
            np.save(
                os.path.join(output_dir, "frame_{:06d}.npy".format(step)),
                frame7)  # (7, H, W): c + eta1 + ... + eta6

            vol = float(cp.mean(c > 0.18))
            mask = cn > 0.18
            labeled, nl = ndimage.label(mask)
            if nl > 0:
                areas = ndimage.sum(mask, labeled, range(1, nl + 1))
                rm = np.sqrt(np.mean(areas) / np.pi) * dx_real * 1e9
            else:
                rm = 0.0

            pg = get_phi(etas)
            gx = (cp.roll(pg, -1, 1) - cp.roll(pg, 1, 1)) * 0.5
            gy = (cp.roll(pg, -1, 0) - cp.roll(pg, 1, 0)) * 0.5
            gl = float(0.5 * cp.sum(cp.sqrt(gx**2 + gy**2))) * dx_real * 1e9

            t_hist.append(float(step * dt))
            r_hist.append(float(rm))
            v_hist.append(float(vol))
            l_hist.append(float(gl))
            tqdm.write("T*={:.0f} r={:.1f}nm vol={:.3f} Lgb={:.0f}nm".format(
                       step * dt, rm, vol, gl))

    dp = os.path.join(output_dir, "diagnostic_data.txt")
    np.savetxt(dp,
               np.column_stack([np.array(t_hist),
                                np.array(r_hist),
                                np.array(v_hist),
                                np.array(l_hist)]),
               header="T* r_mean(nm) vol_frac L_gb(nm)",
               fmt="%.6f %.4f %.4f %.1f")
    print("Run{:02d} 模拟计算完成! 正在写入最后的数据: {}".format(run_id, output_dir))

    # ---- 防丢补丁：跑完立刻打包 ----
    from IPython.display import display, FileLink

    print("📦 正在后台将数据打包为 zip，请稍等几秒钟...")
    zip_filename = "Temporal_dt2p5e-4_Run_{:02d}.zip".format(run_id)
    zip_filepath = os.path.join("/kaggle/working", zip_filename)
    import shutil
    archive_base = os.path.splitext(zip_filepath)[0]
    shutil.make_archive(archive_base, "zip", root_dir=os.path.dirname(output_dir), base_dir=os.path.basename(output_dir))

    if os.path.exists(zip_filepath):
        print("✨ 打包成功！点击下方蓝色链接下载：")
        os.chdir('/kaggle/working')
        display(FileLink(zip_filename))
    else:
        print("❌ 警告：打包似乎失败了，请检查输出日志。")

print("=" * 60)
print("全部 {} 组数据集生成完毕!".format(RUN_END - RUN_START + 1))
print("=" * 60)